In [1]:
# Run once if needed, then restart the kernel if Jupyter asks you to.
!pip install -U openai pandas tqdm python-dotenv

     |████████████████████████████████| 1.3 MB 4.5 MB/s            
  Attempting uninstall: openai
    Found existing installation: openai 2.33.0
    Uninstalling openai-2.33.0:
      Successfully uninstalled openai-2.33.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llama-index-llms-openai 0.3.38 requires openai<2.0.0,>=1.66.3, but you have openai 2.37.0 which is incompatible.


In [2]:
from __future__ import annotations

import os
import re
import json
import time
import uuid
import hashlib
import datetime as dt
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# -----------------------------
# OpenAI client
# -----------------------------
# assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY. Put it in your environment or .env file."
# client = OpenAI()
client = OpenAI(api_key="")

# -----------------------------
# Experiment configuration
# -----------------------------
PROVIDER = "openai"
MODEL_NAME = "gpt-5.4"
TEMPERATURE = 1.0

# Cost-control options.
REASONING_EFFORT = "none"
TEXT_VERBOSITY = "medium"
PROMPT_CACHE_RETENTION = "24h"
PROMPT_CACHE_KEY = None

N_BASE_AGENTS = 150
N_DYADS = 75
N_TRIADS = 50

# Output ceilings by task family.
MAX_OUTPUT_TOKENS_BY_FAMILY = {
    "slogan": 60,
    "aut": 120,
    "story": 700,
}

# A new run_id prevents accidental overwrites.
RUN_ID = dt.datetime.now().strftime("%Y%m%d_%H%M%S") + "__" + uuid.uuid4().hex[:8]

DATA_ROOT = Path("ai_data") / "deflect_creativity" / PROVIDER / f"model_{MODEL_NAME}" / f"run_{RUN_ID}"

DIRS = {
    "metadata": DATA_ROOT / "00_metadata",
    "round1_plans": DATA_ROOT / "01_round1" / "plans",
    "round1_batch_inputs": DATA_ROOT / "01_round1" / "batch_inputs",
    "round1_manifests": DATA_ROOT / "01_round1" / "manifests",
    "round1_raw_outputs": DATA_ROOT / "01_round1" / "raw_outputs",
    "round1_raw_errors": DATA_ROOT / "01_round1" / "raw_errors",
    "round1_parsed": DATA_ROOT / "01_round1" / "parsed",
    "round2_plans": DATA_ROOT / "02_round2" / "plans",
    "round2_batch_inputs": DATA_ROOT / "02_round2" / "batch_inputs",
    "round2_manifests": DATA_ROOT / "02_round2" / "manifests",
    "round2_raw_outputs": DATA_ROOT / "02_round2" / "raw_outputs",
    "round2_raw_errors": DATA_ROOT / "02_round2" / "raw_errors",
    "round2_parsed": DATA_ROOT / "02_round2" / "parsed",
    "compiled": DATA_ROOT / "03_compiled",
}

for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print("Run directory:")
print(DATA_ROOT)

Run directory:
ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f


In [3]:
def now_iso() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat()


def write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def read_json(path: Path) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def append_jsonl(path: Path, record: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def read_jsonl(path: Path) -> list[dict]:
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def safe_slug(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text


def stable_hash(text: str, n: int = 16) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:n]


def clean_model_text(text: Optional[str]) -> Optional[str]:
    if text is None:
        return None
    text = str(text).strip()
    # Remove surrounding quotes if the entire output is quoted.
    if len(text) >= 2 and ((text[0] == text[-1] == '"') or (text[0] == text[-1] == "'")):
        text = text[1:-1].strip()
    return text

In [4]:
run_config = {
    "provider": PROVIDER,
    "model_name": MODEL_NAME,
    "temperature": TEMPERATURE,
    "reasoning_effort": REASONING_EFFORT,
    "text_verbosity": TEXT_VERBOSITY,
    "n_base_agents": N_BASE_AGENTS,
    "n_dyads": N_DYADS,
    "n_triads": N_TRIADS,
    "max_output_tokens_by_family": MAX_OUTPUT_TOKENS_BY_FAMILY,
    "run_id": RUN_ID,
    "data_root": str(DATA_ROOT),
    "created_at_utc": now_iso(),
}

config_path = DIRS["metadata"] / f"experiment_config__{RUN_ID}.json"
write_json(config_path, run_config)

config_path

PosixPath('ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/00_metadata/experiment_config__20260517_204221__2229089f.json')

In [5]:
TASK_SETTINGS = [
    {
        "task_id": "slogan_smartphone",
        "task_family": "slogan",
        "task_label": "Smartphone slogan",
        "task_prompt_key": "smartphone",
    },
    {
        "task_id": "slogan_soda",
        "task_family": "slogan",
        "task_label": "Soda slogan",
        "task_prompt_key": "soda",
    },
    {
        "task_id": "aut_shoe",
        "task_family": "aut",
        "task_label": "AUT shoe",
        "task_prompt_key": "shoe",
        "object": "shoe",
        "common_use": "used as footwear",
    },
    {
        "task_id": "aut_button",
        "task_family": "aut",
        "task_label": "AUT button",
        "task_prompt_key": "button",
        "object": "button",
        "common_use": "used to fasten things",
    },
    {
        "task_id": "story_jungle",
        "task_family": "story",
        "task_label": "Jungle adventure story",
        "task_prompt_key": "jungle",
    },
    {
        "task_id": "story_parachute",
        "task_family": "story",
        "task_label": "Parachute story",
        "task_prompt_key": "parachute",
    },
]

STRATEGIES = ["vanilla", "diverge"]
CONDITIONS = ["base", "dyad", "triad"]

tasks_df = pd.DataFrame(TASK_SETTINGS)
tasks_df

,task_id,task_family,task_label,task_prompt_key,object,common_use
0,slogan_smartphone,slogan,Smartphone slogan,smartphone,NaN,NaN
1,slogan_soda,slogan,Soda slogan,soda,NaN,NaN
2,aut_shoe,aut,AUT shoe,shoe,shoe,used as footwear
3,aut_button,aut,AUT button,button,button,used to fasten things
4,story_jungle,story,Jungle adventure story,jungle,NaN,NaN
5,story_parachute,story,Parachute story,parachute,NaN,NaN


In [6]:
SYSTEM_INSTRUCTIONS = (
    "You are participating in a controlled creativity experiment. "
    "Follow the task instructions exactly. Return exactly one response. "
    "Do not explain your reasoning. Do not include commentary before or after the response."
)


def strategy_block(strategy: str) -> str:
    if strategy == "vanilla":
        return (
            "Creativity goal:\n"
            "- Make the response novel and appropriate for the task."
        )

    if strategy == "diverge":
        return (
            "Creativity goal:\n"
            "- Make the response novel and appropriate for the task.\n"
            "- Try to make it stand out from other responses that might be generated for this same task."
        )

    raise ValueError(f"Unknown strategy: {strategy}")


def base_task_prompt(task: dict) -> str:
    task_id = task["task_id"]

    if task_id == "slogan_smartphone":
        return (
            "You are part of the marketing team at a tech company preparing to launch a new smartphone.\n\n"
            "Generate exactly one marketing slogan for this brand-new smartphone.\n\n"
            "Requirements:\n"
            "- The slogan must not exceed 6 words.\n"
            "- The slogan must be written in English.\n"
            "- You may assume any detail about the smartphone.\n"
            "- Do not list multiple slogans.\n"
            "- Return only the slogan text."
        )

    if task_id == "slogan_soda":
        return (
            "You are part of the marketing team at a beverage company preparing to launch a new soda.\n\n"
            "Generate exactly one marketing slogan for this brand-new soda.\n\n"
            "Requirements:\n"
            "- The slogan must not exceed 6 words.\n"
            "- The slogan must be written in English.\n"
            "- You may assume any detail about the soda.\n"
            "- Do not list multiple slogans.\n"
            "- Return only the slogan text."
        )

    if task_id in {"aut_shoe", "aut_button"}:
        return (
            "You are participating in a creativity task.\n\n"
            f"Object: {task['object']}\n"
            f"Common use to avoid: {task['common_use']}\n\n"
            "Generate exactly one unusual, novel, and plausible alternative use for the object or one of its parts.\n\n"
            "Requirements:\n"
            "- Do not use the common use.\n"
            "- Do not list multiple uses.\n"
            "- The response must be written in English.\n"
            "- Return only the alternative use as a short phrase or one sentence."
        )

    if task_id == "story_jungle":
        return (
            "You are participating in a creative writing task.\n\n"
            "Write exactly one story about an adventure in the jungle.\n\n"
            "Requirements:\n"
            "- The story must be exactly 8 sentences long.\n"
            "- The story must be written in English.\n"
            "- The story must be appropriate for a teenage and young adult audience, approximately ages 15 to 24.\n"
            "- Do not provide multiple story ideas.\n"
            "- Do not summarize the story.\n"
            "- Return only the story."
        )

    if task_id == "story_parachute":
        return (
            "You are participating in a creative writing task.\n\n"
            "Write exactly one story based on this prompt:\n"
            "The parachute isn’t opening up.\n\n"
            "Requirements:\n"
            "- The story must be exactly 8 sentences long.\n"
            "- The story must be written in English.\n"
            "- The story must be appropriate for a teenage and young adult audience, approximately ages 15 to 24.\n"
            "- Do not provide multiple story ideas.\n"
            "- Do not summarize the story.\n"
            "- Return only the story."
        )

    raise ValueError(f"Unknown task_id: {task_id}")


def build_round1_prompt(task: dict, strategy: str) -> str:
    return base_task_prompt(task) + "\n\n" + strategy_block(strategy)


def build_round2_context(
    strategy: str,
    condition: str,
    self_round1: str,
    peer_round1_texts: list[str],
) -> str:
    if condition == "base":
        context = (
            f'Previous response from your first round:\n'
            f'"{self_round1}"\n\n'
        )
    elif condition == "dyad":
        assert len(peer_round1_texts) == 1
        context = (
            f'Previous response from your first round:\n'
            f'"{self_round1}"\n\n'
            f'Previous response from another agent in the same first round:\n'
            f'"{peer_round1_texts[0]}"\n\n'
        )
    elif condition == "triad":
        assert len(peer_round1_texts) == 2
        context = (
            f'Previous response from your first round:\n'
            f'"{self_round1}"\n\n'
            f'Previous responses from two other agents in the same first round:\n'
            f'1. "{peer_round1_texts[0]}"\n'
            f'2. "{peer_round1_texts[1]}"\n\n'
        )
    else:
        raise ValueError(f"Unknown condition: {condition}")

    if strategy == "vanilla":
        return context + "Now generate one new response for the same task."

    if strategy == "diverge":
        return (
            context
            + "Now generate one new response for the same task. "
              "It should stand out from the previous response(s) shown above while still satisfying all task requirements."
        )

    raise ValueError(f"Unknown strategy: {strategy}")


def build_round2_prompt(
    task: dict,
    strategy: str,
    condition: str,
    self_round1: str,
    peer_round1_texts: list[str],
) -> str:
    return (
        base_task_prompt(task)
        + "\n\n"
        + strategy_block(strategy)
        + "\n\n"
        + build_round2_context(
            strategy=strategy,
            condition=condition,
            self_round1=self_round1,
            peer_round1_texts=peer_round1_texts,
        )
    )

In [7]:
def build_agent_roster() -> pd.DataFrame:
    rows = []

    # Base: 150 singleton agents.
    for i in range(1, N_BASE_AGENTS + 1):
        rows.append({
            "condition": "base",
            "group_id": f"base_{i:03d}",
            "group_size": 1,
            "agent_index": 1,
            "agent_id": f"base_{i:03d}__a1",
        })

    # Dyad: 75 dyads x 2 agents = 150 agents.
    for g in range(1, N_DYADS + 1):
        for a in [1, 2]:
            rows.append({
                "condition": "dyad",
                "group_id": f"dyad_{g:03d}",
                "group_size": 2,
                "agent_index": a,
                "agent_id": f"dyad_{g:03d}__a{a}",
            })

    # Triad: 50 triads x 3 agents = 150 agents.
    for g in range(1, N_TRIADS + 1):
        for a in [1, 2, 3]:
            rows.append({
                "condition": "triad",
                "group_id": f"triad_{g:03d}",
                "group_size": 3,
                "agent_index": a,
                "agent_id": f"triad_{g:03d}__a{a}",
            })

    return pd.DataFrame(rows)


agents_df = build_agent_roster()

print(agents_df.shape)
display(agents_df.groupby("condition").agg(
    n_agents=("agent_id", "count"),
    n_groups=("group_id", "nunique"),
    group_size=("group_size", "first"),
))
agents_df.head()

(450, 5)


,n_agents,n_groups,group_size
condition,,,
base,150,150,1
dyad,150,75,2
triad,150,50,3


,condition,group_id,group_size,agent_index,agent_id
0,base,base_001,1,1,base_001__a1
1,base,base_002,1,1,base_002__a1
2,base,base_003,1,1,base_003__a1
3,base,base_004,1,1,base_004__a1
4,base,base_005,1,1,base_005__a1


In [8]:
def build_round1_plan() -> pd.DataFrame:
    rows = []

    for task in TASK_SETTINGS:
        for strategy in STRATEGIES:
            for _, agent in agents_df.iterrows():
                user_prompt = build_round1_prompt(task, strategy)
                request_basis = {
                    "provider": PROVIDER,
                    "model": MODEL_NAME,
                    "round": 1,
                    "task_id": task["task_id"],
                    "task_family": task["task_family"],
                    "strategy": strategy,
                    "condition": agent["condition"],
                    "group_id": agent["group_id"],
                    "agent_id": agent["agent_id"],
                    "agent_index": int(agent["agent_index"]),
                }
                request_key = "r1__" + stable_hash(json.dumps(request_basis, sort_keys=True), 24)

                rows.append({
                    **request_basis,
                    "request_key": request_key,
                    "system_instructions": SYSTEM_INSTRUCTIONS,
                    "user_prompt": user_prompt,
                    "temperature": TEMPERATURE,
                    "max_output_tokens": MAX_OUTPUT_TOKENS_BY_FAMILY[task["task_family"]],
                    "created_at_utc": now_iso(),
                })

    plan_df = pd.DataFrame(rows)

    if plan_df["request_key"].duplicated().any():
        dupes = plan_df[plan_df["request_key"].duplicated(keep=False)].sort_values("request_key")
        raise ValueError(f"Duplicate request_key detected:\n{dupes.head()}")

    return plan_df


round1_plan_df = build_round1_plan()

print(round1_plan_df.shape)
display(round1_plan_df.groupby(["task_id", "strategy", "condition"]).size().reset_index(name="n"))
round1_plan_df.head()

(5400, 16)


,task_id,strategy,condition,n
0,aut_button,diverge,base,150
1,aut_button,diverge,dyad,150
2,aut_button,diverge,triad,150
3,aut_button,vanilla,base,150
4,aut_button,vanilla,dyad,150
5,aut_button,vanilla,triad,150
6,aut_shoe,diverge,base,150
7,aut_shoe,diverge,dyad,150
8,aut_shoe,diverge,triad,150
9,aut_shoe,vanilla,base,150


,provider,model,round,task_id,task_family,strategy,condition,group_id,agent_id,agent_index,request_key,system_instructions,user_prompt,temperature,max_output_tokens,created_at_utc
0,openai,gpt-5.4,1,slogan_smartphone,slogan,vanilla,base,base_001,base_001__a1,1,r1__ae37e6a4459629b922a98b61,You are participating in a controlled creativi...,You are part of the marketing team at a tech c...,1.0,60,2026-05-18T00:42:38.126101+00:00
1,openai,gpt-5.4,1,slogan_smartphone,slogan,vanilla,base,base_002,base_002__a1,1,r1__8cb17f81c524a85271342c9d,You are participating in a controlled creativi...,You are part of the marketing team at a tech c...,1.0,60,2026-05-18T00:42:38.126248+00:00
2,openai,gpt-5.4,1,slogan_smartphone,slogan,vanilla,base,base_003,base_003__a1,1,r1__57bc8ea5d603182f310443f0,You are participating in a controlled creativi...,You are part of the marketing team at a tech c...,1.0,60,2026-05-18T00:42:38.126362+00:00
3,openai,gpt-5.4,1,slogan_smartphone,slogan,vanilla,base,base_004,base_004__a1,1,r1__f0ef49354704a11502587f82,You are participating in a controlled creativi...,You are part of the marketing team at a tech c...,1.0,60,2026-05-18T00:42:38.126472+00:00
4,openai,gpt-5.4,1,slogan_smartphone,slogan,vanilla,base,base_005,base_005__a1,1,r1__dd444f22e6e3df007e25cf4e,You are participating in a controlled creativi...,You are part of the marketing team at a tech c...,1.0,60,2026-05-18T00:42:38.126581+00:00


In [9]:
def make_openai_responses_batch_jsonl(
    plan_df: pd.DataFrame,
    round_name: str,
    output_dir: Path,
) -> tuple[Path, Path]:
    timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    stem = f"{round_name}__{PROVIDER}__{MODEL_NAME}__{timestamp}"

    plan_path = output_dir.parent / "plans" / f"{stem}__plan.csv"
    jsonl_path = output_dir / f"{stem}__batch_input.jsonl"

    plan_path.parent.mkdir(parents=True, exist_ok=True)
    jsonl_path.parent.mkdir(parents=True, exist_ok=True)

    plan_df.to_csv(plan_path, index=False)

    with open(jsonl_path, "w", encoding="utf-8") as f:
        for _, row in plan_df.iterrows():
            body = {
                "model": MODEL_NAME,
                "instructions": row["system_instructions"],
                "input": row["user_prompt"],
                "temperature": float(row["temperature"]),
                "max_output_tokens": int(row["max_output_tokens"]),
            }

            # Optional knobs for GPT-5-style models, only if explicitly enabled above.
            if REASONING_EFFORT is not None:
                body["reasoning"] = {"effort": REASONING_EFFORT}

            if TEXT_VERBOSITY is not None:
                body["text"] = {"verbosity": TEXT_VERBOSITY}

            if PROMPT_CACHE_RETENTION is not None:
                body["prompt_cache_retention"] = PROMPT_CACHE_RETENTION

            if PROMPT_CACHE_KEY is not None:
                body["prompt_cache_key"] = PROMPT_CACHE_KEY
    

            request = {
                "custom_id": row["request_key"],
                "method": "POST",
                "url": "/v1/responses",
                "body": body,
            }
            f.write(json.dumps(request, ensure_ascii=False) + "\n")

    print(f"Wrote plan:  {plan_path}")
    print(f"Wrote batch: {jsonl_path}")
    print(f"Requests:    {len(plan_df):,}")

    return jsonl_path, plan_path


round1_jsonl_path, round1_plan_path = make_openai_responses_batch_jsonl(
    plan_df=round1_plan_df,
    round_name="round1",
    output_dir=DIRS["round1_batch_inputs"],
)

round1_jsonl_path, round1_plan_path

Wrote plan:  ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/01_round1/plans/round1__openai__gpt-5.4__20260517_204246__plan.csv
Wrote batch: ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/01_round1/batch_inputs/round1__openai__gpt-5.4__20260517_204246__batch_input.jsonl
Requests:    5,400


(PosixPath('ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/01_round1/batch_inputs/round1__openai__gpt-5.4__20260517_204246__batch_input.jsonl'),
 PosixPath('ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/01_round1/plans/round1__openai__gpt-5.4__20260517_204246__plan.csv'))

In [10]:
def submit_openai_batch(
    batch_jsonl_path: Path,
    round_name: str,
    plan_path: Path,
    manifest_dir: Path,
) -> dict:
    batch_input_file = client.files.create(
        file=open(batch_jsonl_path, "rb"),
        purpose="batch",
    )

    batch = client.batches.create(
        input_file_id=batch_input_file.id,
        endpoint="/v1/responses",
        completion_window="24h",
        metadata={
            "project": "deflect_creativity",
            "round": round_name,
            "provider": PROVIDER,
            "model": MODEL_NAME,
            "run_id": RUN_ID,
            "local_input_file": str(batch_jsonl_path),
            "local_plan_file": str(plan_path),
        },
    )

    batch_info = {
        "run_id": RUN_ID,
        "round": round_name,
        "provider": PROVIDER,
        "model": MODEL_NAME,
        "batch_id": batch.id,
        "input_file_id": batch_input_file.id,
        "status_at_submission": batch.status,
        "submitted_at_utc": now_iso(),
        "batch_jsonl_path": str(batch_jsonl_path),
        "plan_path": str(plan_path),
        "data_root": str(DATA_ROOT),
    }

    manifest_path = manifest_dir / f"{round_name}__{PROVIDER}__{MODEL_NAME}__batch_manifest__{batch.id}.json"
    write_json(manifest_path, batch_info)
    batch_info["manifest_path"] = str(manifest_path)

    print("Submitted batch:")
    print(json.dumps(batch_info, indent=2))

    return batch_info


round1_batch_info = submit_openai_batch(
    batch_jsonl_path=round1_jsonl_path,
    round_name="round1",
    plan_path=round1_plan_path,
    manifest_dir=DIRS["round1_manifests"],
)

round1_batch_info

Submitted batch:
{
  "run_id": "20260517_204221__2229089f",
  "round": "round1",
  "provider": "openai",
  "model": "gpt-5.4",
  "batch_id": "batch_6a0a608e99408190b15d321761c983a7",
  "input_file_id": "file-Do68ktLUUqJii86CP4yogx",
  "status_at_submission": "validating",
  "submitted_at_utc": "2026-05-18T00:42:55.530473+00:00",
  "batch_jsonl_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/01_round1/batch_inputs/round1__openai__gpt-5.4__20260517_204246__batch_input.jsonl",
  "plan_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/01_round1/plans/round1__openai__gpt-5.4__20260517_204246__plan.csv",
  "data_root": "ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f",
  "manifest_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/01_round1/manifests/round1__openai__gpt-5.4__batch_manifest__batch_6a0a608e99408190b15d321761c983a7.json"
}


{'run_id': '20260517_204221__2229089f',
 'round': 'round1',
 'provider': 'openai',
 'model': 'gpt-5.4',
 'batch_id': 'batch_6a0a608e99408190b15d321761c983a7',
 'input_file_id': 'file-Do68ktLUUqJii86CP4yogx',
 'status_at_submission': 'validating',
 'submitted_at_utc': '2026-05-18T00:42:55.530473+00:00',
 'batch_jsonl_path': 'ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/01_round1/batch_inputs/round1__openai__gpt-5.4__20260517_204246__batch_input.jsonl',
 'plan_path': 'ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/01_round1/plans/round1__openai__gpt-5.4__20260517_204246__plan.csv',
 'data_root': 'ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f',
 'manifest_path': 'ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/01_round1/manifests/round1__openai__gpt-5.4__batch_manifest__batch_6a0a608e99408190b15d321761c983a7.json'}

In [27]:
def check_openai_batch(batch_id: str) -> dict:
    batch = client.batches.retrieve(batch_id)

    info = {
        "batch_id": batch.id,
        "status": batch.status,
        "created_at": batch.created_at,
        "in_progress_at": getattr(batch, "in_progress_at", None),
        "finalizing_at": getattr(batch, "finalizing_at", None),
        "completed_at": getattr(batch, "completed_at", None),
        "failed_at": getattr(batch, "failed_at", None),
        "expired_at": getattr(batch, "expired_at", None),
        "cancelled_at": getattr(batch, "cancelled_at", None),
        "request_counts": None,
        "output_file_id": batch.output_file_id,
        "error_file_id": batch.error_file_id,
    }

    if batch.request_counts:
        info["request_counts"] = {
            "total": batch.request_counts.total,
            "completed": batch.request_counts.completed,
            "failed": batch.request_counts.failed,
        }

    # Usage fields may or may not be populated depending on SDK/API version.
    usage = getattr(batch, "usage", None)
    if usage:
        try:
            info["usage"] = usage.model_dump()
        except Exception:
            info["usage"] = str(usage)

    print(json.dumps(info, indent=2))
    return info


round1_status = check_openai_batch(round1_batch_info["batch_id"])
round1_status

{
  "batch_id": "batch_6a0a608e99408190b15d321761c983a7",
  "status": "completed",
  "created_at": 1779064974,
  "in_progress_at": 1779065040,
  "finalizing_at": 1779066074,
  "completed_at": 1779066558,
  "failed_at": null,
  "expired_at": null,
  "cancelled_at": null,
  "request_counts": {
    "total": 5400,
    "completed": 5400,
    "failed": 0
  },
  "output_file_id": "file-Gk4WzTYzdGwmU5t2X4fcAC",
  "error_file_id": null,
  "usage": {
    "input_tokens": 819900,
    "input_tokens_details": {
      "cached_tokens": 0
    },
    "output_tokens": 547904,
    "output_tokens_details": {
      "reasoning_tokens": 0
    },
    "total_tokens": 1367804
  }
}


{'batch_id': 'batch_6a0a608e99408190b15d321761c983a7',
 'status': 'completed',
 'created_at': 1779064974,
 'in_progress_at': 1779065040,
 'finalizing_at': 1779066074,
 'completed_at': 1779066558,
 'failed_at': None,
 'expired_at': None,
 'cancelled_at': None,
 'request_counts': {'total': 5400, 'completed': 5400, 'failed': 0},
 'output_file_id': 'file-Gk4WzTYzdGwmU5t2X4fcAC',
 'error_file_id': None,
 'usage': {'input_tokens': 819900,
  'input_tokens_details': {'cached_tokens': 0},
  'output_tokens': 547904,
  'output_tokens_details': {'reasoning_tokens': 0},
  'total_tokens': 1367804}}

In [ ]:
# Optional reload after restart:
# Replace with the manifest path printed in Cell 10 if needed.

# manifest_path = Path("ai_data/deflect_creativity/openai/model_gpt-5.4/run_.../01_round1/manifests/round1__openai__gpt-5.4__batch_manifest__batch_....json")
# round1_batch_info = read_json(manifest_path)
# DATA_ROOT = Path(round1_batch_info["data_root"])
# round1_batch_info

In [28]:
def download_openai_batch_results(
    batch_id: str,
    raw_output_dir: Path,
    raw_error_dir: Path,
    round_name: str,
) -> tuple[Optional[Path], Optional[Path]]:
    batch = client.batches.retrieve(batch_id)

    if batch.status != "completed":
        print(f"Batch is not completed yet. Current status: {batch.status}")
        return None, None

    output_path = raw_output_dir / f"{round_name}__{batch_id}__output.jsonl"
    error_path = raw_error_dir / f"{round_name}__{batch_id}__errors.jsonl"

    if batch.output_file_id:
        file_response = client.files.content(batch.output_file_id)
        output_path.write_text(file_response.text, encoding="utf-8")
        print(f"Downloaded output: {output_path}")

    if batch.error_file_id:
        error_response = client.files.content(batch.error_file_id)
        error_path.write_text(error_response.text, encoding="utf-8")
        print(f"Downloaded errors: {error_path}")
    else:
        error_path = None
        print("No error file.")

    return output_path, error_path


round1_output_path, round1_error_path = download_openai_batch_results(
    batch_id=round1_batch_info["batch_id"],
    raw_output_dir=DIRS["round1_raw_outputs"],
    raw_error_dir=DIRS["round1_raw_errors"],
    round_name="round1",
)

round1_output_path, round1_error_path

Downloaded output: ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/01_round1/raw_outputs/round1__batch_6a0a608e99408190b15d321761c983a7__output.jsonl
No error file.


(PosixPath('ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/01_round1/raw_outputs/round1__batch_6a0a608e99408190b15d321761c983a7__output.jsonl'),
 None)

In [29]:
def extract_text_from_responses_api_body(body: dict) -> str:
    if not isinstance(body, dict):
        return ""

    if body.get("output_text"):
        return str(body["output_text"]).strip()

    texts = []
    for item in body.get("output", []) or []:
        for content in item.get("content", []) or []:
            if isinstance(content, dict) and content.get("type") in {"output_text", "text"} and "text" in content:
                texts.append(content["text"])

    return "\n".join(texts).strip()


def parse_openai_batch_output_to_standard_files(
    batch_output_path: Path,
    plan_path: Path,
    parsed_dir: Path,
    round_name: str,
    batch_id: str,
) -> dict:
    plan_df = pd.read_csv(plan_path)
    plan_by_key = {
        row["request_key"]: row.to_dict()
        for _, row in plan_df.iterrows()
    }

    batch_records = read_jsonl(batch_output_path)

    parsed_jsonl_path = parsed_dir / f"{round_name}__{PROVIDER}__{MODEL_NAME}__{batch_id}__parsed.jsonl"
    parsed_csv_path = parsed_dir / f"{round_name}__{PROVIDER}__{MODEL_NAME}__{batch_id}__parsed.csv"
    parsed_pkl_path = parsed_dir / f"{round_name}__{PROVIDER}__{MODEL_NAME}__{batch_id}__parsed.pkl"

    if parsed_jsonl_path.exists():
        raise FileExistsError(f"Refusing to overwrite existing parsed JSONL: {parsed_jsonl_path}")

    parsed_records = []
    n_success = 0
    n_empty = 0
    n_error = 0

    for rec in batch_records:
        request_key = rec.get("custom_id")
        plan_row = plan_by_key.get(request_key, {})
        response = rec.get("response") or {}
        error = rec.get("error")

        if response and response.get("body"):
            body = response["body"]
            text = clean_model_text(extract_text_from_responses_api_body(body))
            usage = body.get("usage")

            status = "success" if text else "empty_text"
            n_success += int(status == "success")
            n_empty += int(status == "empty_text")

            record = {
                **plan_row,
                "parsed_at_utc": now_iso(),
                "status": status,
                "text": text,
                "provider_response_id": body.get("id"),
                "usage": usage,
                "error": None if text else "No text extracted from response body.",
                "batch_custom_id": request_key,
                "batch_output_file": str(batch_output_path),
                "batch_id": batch_id,
            }
        else:
            n_error += 1
            record = {
                **plan_row,
                "parsed_at_utc": now_iso(),
                "status": "error",
                "text": None,
                "provider_response_id": None,
                "usage": None,
                "error": error,
                "batch_custom_id": request_key,
                "batch_output_file": str(batch_output_path),
                "batch_id": batch_id,
            }

        parsed_records.append(record)
        append_jsonl(parsed_jsonl_path, record)

    parsed_df = pd.DataFrame(parsed_records)
    parsed_df.to_csv(parsed_csv_path, index=False)
    parsed_df.to_pickle(parsed_pkl_path)

    summary = {
        "round": round_name,
        "batch_id": batch_id,
        "n_records": len(parsed_df),
        "n_success": n_success,
        "n_empty_text": n_empty,
        "n_error": n_error,
        "parsed_jsonl_path": str(parsed_jsonl_path),
        "parsed_csv_path": str(parsed_csv_path),
        "parsed_pkl_path": str(parsed_pkl_path),
    }

    summary_path = parsed_dir / f"{round_name}__{PROVIDER}__{MODEL_NAME}__{batch_id}__parse_summary.json"
    write_json(summary_path, summary)

    print(json.dumps(summary, indent=2))
    return summary


round1_parse_summary = parse_openai_batch_output_to_standard_files(
    batch_output_path=round1_output_path,
    plan_path=Path(round1_batch_info["plan_path"]),
    parsed_dir=DIRS["round1_parsed"],
    round_name="round1",
    batch_id=round1_batch_info["batch_id"],
)

round1_df = pd.read_pickle(round1_parse_summary["parsed_pkl_path"])
print(round1_df.shape)
display(round1_df["status"].value_counts(dropna=False))
round1_df.head()

{
  "round": "round1",
  "batch_id": "batch_6a0a608e99408190b15d321761c983a7",
  "n_records": 5400,
  "n_success": 5400,
  "n_empty_text": 0,
  "n_error": 0,
  "parsed_jsonl_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/01_round1/parsed/round1__openai__gpt-5.4__batch_6a0a608e99408190b15d321761c983a7__parsed.jsonl",
  "parsed_csv_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/01_round1/parsed/round1__openai__gpt-5.4__batch_6a0a608e99408190b15d321761c983a7__parsed.csv",
  "parsed_pkl_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/01_round1/parsed/round1__openai__gpt-5.4__batch_6a0a608e99408190b15d321761c983a7__parsed.pkl"
}
(5400, 25)


status
success    5400
Name: count, dtype: int64

,provider,model,round,task_id,task_family,strategy,condition,group_id,agent_id,agent_index,...,created_at_utc,parsed_at_utc,status,text,provider_response_id,usage,error,batch_custom_id,batch_output_file,batch_id
0,openai,gpt-5.4,1,slogan_smartphone,slogan,vanilla,base,base_001,base_001__a1,1,...,2026-05-18T00:42:38.126101+00:00,2026-05-18T01:10:12.375391+00:00,success,"Tomorrow, perfectly in your palm.",resp_055cf08add51d9be006a0a6151cbf4819dbef406d...,"{'input_tokens': 135, 'input_tokens_details': ...",None,r1__ae37e6a4459629b922a98b61,ai_data/deflect_creativity/openai/model_gpt-5....,batch_6a0a608e99408190b15d321761c983a7
1,openai,gpt-5.4,1,slogan_smartphone,slogan,vanilla,base,base_002,base_002__a1,1,...,2026-05-18T00:42:38.126248+00:00,2026-05-18T01:10:12.375767+00:00,success,"Tomorrow, perfectly in your palm.",resp_0ec4095c77c1e854006a0a6154c72081929132d43...,"{'input_tokens': 135, 'input_tokens_details': ...",None,r1__8cb17f81c524a85271342c9d,ai_data/deflect_creativity/openai/model_gpt-5....,batch_6a0a608e99408190b15d321761c983a7
2,openai,gpt-5.4,1,slogan_smartphone,slogan,vanilla,base,base_003,base_003__a1,1,...,2026-05-18T00:42:38.126362+00:00,2026-05-18T01:10:12.375865+00:00,success,"Pocket the Future, Effortlessly.",resp_0299eeb61c4c8903006a0a60f8a06081a39225b4a...,"{'input_tokens': 135, 'input_tokens_details': ...",None,r1__57bc8ea5d603182f310443f0,ai_data/deflect_creativity/openai/model_gpt-5....,batch_6a0a608e99408190b15d321761c983a7
3,openai,gpt-5.4,1,slogan_smartphone,slogan,vanilla,base,base_004,base_004__a1,1,...,2026-05-18T00:42:38.126472+00:00,2026-05-18T01:10:12.375970+00:00,success,"Tomorrow, perfectly in your pocket.",resp_0aee229f462ce6e3006a0a61340574819db4fd948...,"{'input_tokens': 135, 'input_tokens_details': ...",None,r1__f0ef49354704a11502587f82,ai_data/deflect_creativity/openai/model_gpt-5....,batch_6a0a608e99408190b15d321761c983a7
4,openai,gpt-5.4,1,slogan_smartphone,slogan,vanilla,base,base_005,base_005__a1,1,...,2026-05-18T00:42:38.126581+00:00,2026-05-18T01:10:12.376036+00:00,success,"Tomorrow, held in your hand.",resp_0063573e3ae1bee0006a0a60f8d44c81929f6de37...,"{'input_tokens': 135, 'input_tokens_details': ...",None,r1__dd444f22e6e3df007e25cf4e,ai_data/deflect_creativity/openai/model_gpt-5....,batch_6a0a608e99408190b15d321761c983a7


In [30]:
expected_round1 = len(TASK_SETTINGS) * len(STRATEGIES) * len(agents_df)
actual_round1 = len(round1_df)

print("Expected Round 1 rows:", expected_round1)
print("Actual Round 1 rows:  ", actual_round1)

display(round1_df.groupby(["task_id", "strategy", "condition", "status"]).size().reset_index(name="n"))

if actual_round1 != expected_round1:
    print("WARNING: Row count mismatch. Inspect errors before proceeding.")

if (round1_df["status"] != "success").any():
    print("WARNING: Some Round 1 calls failed or returned empty text. Inspect before proceeding to Round 2.")
    display(round1_df[round1_df["status"] != "success"].head(20))
else:
    print("Round 1 looks complete.")

Expected Round 1 rows: 5400
Actual Round 1 rows:   5400


,task_id,strategy,condition,status,n
0,aut_button,diverge,base,success,150
1,aut_button,diverge,dyad,success,150
2,aut_button,diverge,triad,success,150
3,aut_button,vanilla,base,success,150
4,aut_button,vanilla,dyad,success,150
5,aut_button,vanilla,triad,success,150
6,aut_shoe,diverge,base,success,150
7,aut_shoe,diverge,dyad,success,150
8,aut_shoe,diverge,triad,success,150
9,aut_shoe,vanilla,base,success,150


Round 1 looks complete.


In [31]:
def build_round1_lookup(round1_df: pd.DataFrame) -> dict:
    success_df = round1_df[round1_df["status"] == "success"].copy()

    required_cols = ["task_id", "strategy", "condition", "group_id", "agent_id"]
    if success_df.duplicated(required_cols).any():
        dupes = success_df[success_df.duplicated(required_cols, keep=False)].sort_values(required_cols)
        raise ValueError(f"Duplicate Round 1 successful records for same agent/task/strategy:\n{dupes[required_cols + ['text']].head()}")

    return {
        (row["task_id"], row["strategy"], row["condition"], row["group_id"], row["agent_id"]): row.to_dict()
        for _, row in success_df.iterrows()
    }


def get_task_by_id(task_id: str) -> dict:
    for t in TASK_SETTINGS:
        if t["task_id"] == task_id:
            return t
    raise KeyError(task_id)


def build_round2_plan(round1_df: pd.DataFrame) -> pd.DataFrame:
    r1_lookup = build_round1_lookup(round1_df)
    rows = []

    # Use successful R1 only. If any failed, rerun/fix before building R2.
    r1_success = round1_df[round1_df["status"] == "success"].copy()

    for (task_id, strategy, condition, group_id), group in r1_success.groupby(
        ["task_id", "strategy", "condition", "group_id"],
        sort=True,
    ):
        task = get_task_by_id(task_id)
        group = group.sort_values("agent_index").copy()

        expected_group_size = {"base": 1, "dyad": 2, "triad": 3}[condition]
        if len(group) != expected_group_size:
            raise ValueError(
                f"Group size mismatch for {(task_id, strategy, condition, group_id)}: "
                f"expected {expected_group_size}, got {len(group)}"
            )

        for _, ego in group.iterrows():
            self_text = ego["text"]

            peer_rows = group[group["agent_id"] != ego["agent_id"]].sort_values("agent_index")
            peer_texts = peer_rows["text"].tolist()
            peer_agent_ids = peer_rows["agent_id"].tolist()

            user_prompt = build_round2_prompt(
                task=task,
                strategy=strategy,
                condition=condition,
                self_round1=self_text,
                peer_round1_texts=peer_texts,
            )

            request_basis = {
                "provider": PROVIDER,
                "model": MODEL_NAME,
                "round": 2,
                "task_id": task_id,
                "task_family": task["task_family"],
                "strategy": strategy,
                "condition": condition,
                "group_id": group_id,
                "agent_id": ego["agent_id"],
                "agent_index": int(ego["agent_index"]),
                "self_round1_request_key": ego["request_key"],
                "peer_round1_agent_ids": "|".join(peer_agent_ids),
            }
            request_key = "r2__" + stable_hash(json.dumps(request_basis, sort_keys=True), 24)

            rows.append({
                **request_basis,
                "request_key": request_key,
                "system_instructions": SYSTEM_INSTRUCTIONS,
                "user_prompt": user_prompt,
                "temperature": TEMPERATURE,
                "max_output_tokens": MAX_OUTPUT_TOKENS_BY_FAMILY[task["task_family"]],
                "self_round1_text": self_text,
                "peer_round1_texts_json": json.dumps(peer_texts, ensure_ascii=False),
                "created_at_utc": now_iso(),
            })

    plan_df = pd.DataFrame(rows)

    if plan_df["request_key"].duplicated().any():
        dupes = plan_df[plan_df["request_key"].duplicated(keep=False)].sort_values("request_key")
        raise ValueError(f"Duplicate request_key detected:\n{dupes.head()}")

    return plan_df


round2_plan_df = build_round2_plan(round1_df)

print(round2_plan_df.shape)
display(round2_plan_df.groupby(["task_id", "strategy", "condition"]).size().reset_index(name="n"))
round2_plan_df.head()

(5400, 20)


,task_id,strategy,condition,n
0,aut_button,diverge,base,150
1,aut_button,diverge,dyad,150
2,aut_button,diverge,triad,150
3,aut_button,vanilla,base,150
4,aut_button,vanilla,dyad,150
5,aut_button,vanilla,triad,150
6,aut_shoe,diverge,base,150
7,aut_shoe,diverge,dyad,150
8,aut_shoe,diverge,triad,150
9,aut_shoe,vanilla,base,150


,provider,model,round,task_id,task_family,strategy,condition,group_id,agent_id,agent_index,self_round1_request_key,peer_round1_agent_ids,request_key,system_instructions,user_prompt,temperature,max_output_tokens,self_round1_text,peer_round1_texts_json,created_at_utc
0,openai,gpt-5.4,2,aut_button,aut,diverge,base,base_001,base_001__a1,1,r1__eeeff5348ceea507a0291cd3,,r2__aa96f4ec51cf9b0a2948445a,You are participating in a controlled creativi...,You are participating in a creativity task.\n\...,1.0,120,A button can be glued under a wobbly table leg...,[],2026-05-18T01:10:17.205165+00:00
1,openai,gpt-5.4,2,aut_button,aut,diverge,base,base_002,base_002__a1,1,r1__2ecc55303e06d1b1b983de33,,r2__be4c9723220ca51d483b00f8,You are participating in a controlled creativi...,You are participating in a creativity task.\n\...,1.0,120,A button can serve as a tiny loom for weaving ...,[],2026-05-18T01:10:17.205509+00:00
2,openai,gpt-5.4,2,aut_button,aut,diverge,base,base_003,base_003__a1,1,r1__bdca0bc782d460428d40b829,,r2__b607b909781a1c9109e5ee1a,You are participating in a controlled creativi...,You are participating in a creativity task.\n\...,1.0,120,A button sewn inside a shirt cuff can serve as...,[],2026-05-18T01:10:17.205823+00:00
3,openai,gpt-5.4,2,aut_button,aut,diverge,base,base_004,base_004__a1,1,r1__61433013f153cb6bbda73844,,r2__0309eb4d8b21f5934d32f4ac,You are participating in a controlled creativi...,You are participating in a creativity task.\n\...,1.0,120,A button can serve as a tiny paint palette for...,[],2026-05-18T01:10:17.206124+00:00
4,openai,gpt-5.4,2,aut_button,aut,diverge,base,base_005,base_005__a1,1,r1__8b386a285bd41aea4087e87b,,r2__288451ff7a28713399c3fe9c,You are participating in a controlled creativi...,You are participating in a creativity task.\n\...,1.0,120,Use a button as a tiny paint palette for mixin...,[],2026-05-18T01:10:17.206424+00:00


In [32]:
round2_jsonl_path, round2_plan_path = make_openai_responses_batch_jsonl(
    plan_df=round2_plan_df,
    round_name="round2",
    output_dir=DIRS["round2_batch_inputs"],
)

round2_jsonl_path, round2_plan_path

Wrote plan:  ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/02_round2/plans/round2__openai__gpt-5.4__20260517_211019__plan.csv
Wrote batch: ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/02_round2/batch_inputs/round2__openai__gpt-5.4__20260517_211019__batch_input.jsonl
Requests:    5,400


(PosixPath('ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/02_round2/batch_inputs/round2__openai__gpt-5.4__20260517_211019__batch_input.jsonl'),
 PosixPath('ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/02_round2/plans/round2__openai__gpt-5.4__20260517_211019__plan.csv'))

In [33]:
round2_batch_info = submit_openai_batch(
    batch_jsonl_path=round2_jsonl_path,
    round_name="round2",
    plan_path=round2_plan_path,
    manifest_dir=DIRS["round2_manifests"],
)

round2_batch_info

Submitted batch:
{
  "run_id": "20260517_204221__2229089f",
  "round": "round2",
  "provider": "openai",
  "model": "gpt-5.4",
  "batch_id": "batch_6a0a66ff75148190916b217bd42d0379",
  "input_file_id": "file-8U6CgFwDEQFGg3Gtm9KxKA",
  "status_at_submission": "validating",
  "submitted_at_utc": "2026-05-18T01:10:24.223380+00:00",
  "batch_jsonl_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/02_round2/batch_inputs/round2__openai__gpt-5.4__20260517_211019__batch_input.jsonl",
  "plan_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/02_round2/plans/round2__openai__gpt-5.4__20260517_211019__plan.csv",
  "data_root": "ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f",
  "manifest_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/02_round2/manifests/round2__openai__gpt-5.4__batch_manifest__batch_6a0a66ff75148190916b217bd42d0379.json"
}


{'run_id': '20260517_204221__2229089f',
 'round': 'round2',
 'provider': 'openai',
 'model': 'gpt-5.4',
 'batch_id': 'batch_6a0a66ff75148190916b217bd42d0379',
 'input_file_id': 'file-8U6CgFwDEQFGg3Gtm9KxKA',
 'status_at_submission': 'validating',
 'submitted_at_utc': '2026-05-18T01:10:24.223380+00:00',
 'batch_jsonl_path': 'ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/02_round2/batch_inputs/round2__openai__gpt-5.4__20260517_211019__batch_input.jsonl',
 'plan_path': 'ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/02_round2/plans/round2__openai__gpt-5.4__20260517_211019__plan.csv',
 'data_root': 'ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f',
 'manifest_path': 'ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/02_round2/manifests/round2__openai__gpt-5.4__batch_manifest__batch_6a0a66ff75148190916b217bd42d0379.json'}

In [41]:
round2_status = check_openai_batch(round2_batch_info["batch_id"])
round2_status

{
  "batch_id": "batch_6a0a66ff75148190916b217bd42d0379",
  "status": "completed",
  "created_at": 1779066623,
  "in_progress_at": 1779066686,
  "finalizing_at": 1779067907,
  "completed_at": 1779068199,
  "failed_at": null,
  "expired_at": null,
  "cancelled_at": null,
  "request_counts": {
    "total": 5400,
    "completed": 5400,
    "failed": 0
  },
  "output_file_id": "file-96cowvMwHSRVjMrVVt9WT8",
  "error_file_id": null,
  "usage": {
    "input_tokens": 2074789,
    "input_tokens_details": {
      "cached_tokens": 0
    },
    "output_tokens": 593395,
    "output_tokens_details": {
      "reasoning_tokens": 0
    },
    "total_tokens": 2668184
  }
}


{'batch_id': 'batch_6a0a66ff75148190916b217bd42d0379',
 'status': 'completed',
 'created_at': 1779066623,
 'in_progress_at': 1779066686,
 'finalizing_at': 1779067907,
 'completed_at': 1779068199,
 'failed_at': None,
 'expired_at': None,
 'cancelled_at': None,
 'request_counts': {'total': 5400, 'completed': 5400, 'failed': 0},
 'output_file_id': 'file-96cowvMwHSRVjMrVVt9WT8',
 'error_file_id': None,
 'usage': {'input_tokens': 2074789,
  'input_tokens_details': {'cached_tokens': 0},
  'output_tokens': 593395,
  'output_tokens_details': {'reasoning_tokens': 0},
  'total_tokens': 2668184}}

In [42]:
# Optional reload after restart:
# Replace with the manifest path printed in Cell 18 if needed.

# manifest_path = Path("ai_data/deflect_creativity/openai/model_gpt-5.4/run_.../02_round2/manifests/round2__openai__gpt-5.4__batch_manifest__batch_....json")
# round2_batch_info = read_json(manifest_path)
# DATA_ROOT = Path(round2_batch_info["data_root"])
# round2_batch_info

In [43]:
round2_output_path, round2_error_path = download_openai_batch_results(
    batch_id=round2_batch_info["batch_id"],
    raw_output_dir=DIRS["round2_raw_outputs"],
    raw_error_dir=DIRS["round2_raw_errors"],
    round_name="round2",
)

round2_output_path, round2_error_path

Downloaded output: ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/02_round2/raw_outputs/round2__batch_6a0a66ff75148190916b217bd42d0379__output.jsonl
No error file.


(PosixPath('ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/02_round2/raw_outputs/round2__batch_6a0a66ff75148190916b217bd42d0379__output.jsonl'),
 None)

In [44]:
round2_parse_summary = parse_openai_batch_output_to_standard_files(
    batch_output_path=round2_output_path,
    plan_path=Path(round2_batch_info["plan_path"]),
    parsed_dir=DIRS["round2_parsed"],
    round_name="round2",
    batch_id=round2_batch_info["batch_id"],
)

round2_df = pd.read_pickle(round2_parse_summary["parsed_pkl_path"])
print(round2_df.shape)
display(round2_df["status"].value_counts(dropna=False))
round2_df.head()

{
  "round": "round2",
  "batch_id": "batch_6a0a66ff75148190916b217bd42d0379",
  "n_records": 5400,
  "n_success": 5400,
  "n_empty_text": 0,
  "n_error": 0,
  "parsed_jsonl_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/02_round2/parsed/round2__openai__gpt-5.4__batch_6a0a66ff75148190916b217bd42d0379__parsed.jsonl",
  "parsed_csv_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/02_round2/parsed/round2__openai__gpt-5.4__batch_6a0a66ff75148190916b217bd42d0379__parsed.csv",
  "parsed_pkl_path": "ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/02_round2/parsed/round2__openai__gpt-5.4__batch_6a0a66ff75148190916b217bd42d0379__parsed.pkl"
}
(5400, 29)


status
success    5400
Name: count, dtype: int64

,provider,model,round,task_id,task_family,strategy,condition,group_id,agent_id,agent_index,...,created_at_utc,parsed_at_utc,status,text,provider_response_id,usage,error,batch_custom_id,batch_output_file,batch_id
0,openai,gpt-5.4,2,aut_button,aut,diverge,base,base_001,base_001__a1,1,...,2026-05-18T01:10:17.205165+00:00,2026-05-18T01:53:54.859355+00:00,success,Sew a button inside a lampshade as a discreet ...,resp_032ec49443caf5b6006a0a678943b481978eac638...,"{'input_tokens': 221, 'input_tokens_details': ...",None,r2__aa96f4ec51cf9b0a2948445a,ai_data/deflect_creativity/openai/model_gpt-5....,batch_6a0a66ff75148190916b217bd42d0379
1,openai,gpt-5.4,2,aut_button,aut,diverge,base,base_002,base_002__a1,1,...,2026-05-18T01:10:17.205509+00:00,2026-05-18T01:53:54.859775+00:00,success,A button can be embedded in candle wax as a hi...,resp_0192f95fdae9e134006a0a678bd60c81948e878dc...,"{'input_tokens': 216, 'input_tokens_details': ...",None,r2__be4c9723220ca51d483b00f8,ai_data/deflect_creativity/openai/model_gpt-5....,batch_6a0a66ff75148190916b217bd42d0379
2,openai,gpt-5.4,2,aut_button,aut,diverge,base,base_003,base_003__a1,1,...,2026-05-18T01:10:17.205823+00:00,2026-05-18T01:53:54.860294+00:00,success,A button tied to a houseplant stem works as a ...,resp_0affdd9f258817dd006a0a678ca2d08195923ebfc...,"{'input_tokens': 225, 'input_tokens_details': ...",None,r2__b607b909781a1c9109e5ee1a,ai_data/deflect_creativity/openai/model_gpt-5....,batch_6a0a66ff75148190916b217bd42d0379
3,openai,gpt-5.4,2,aut_button,aut,diverge,base,base_004,base_004__a1,1,...,2026-05-18T01:10:17.206124+00:00,2026-05-18T01:53:54.860427+00:00,success,A button can be glued under a wobbly picture f...,resp_08319c129b48509a006a0a678ca9888197941c779...,"{'input_tokens': 219, 'input_tokens_details': ...",None,r2__0309eb4d8b21f5934d32f4ac,ai_data/deflect_creativity/openai/model_gpt-5....,batch_6a0a66ff75148190916b217bd42d0379
4,openai,gpt-5.4,2,aut_button,aut,diverge,base,base_005,base_005__a1,1,...,2026-05-18T01:10:17.206424+00:00,2026-05-18T01:53:54.860554+00:00,success,Glue a button inside a flowerpot as a hidden r...,resp_05ecbf4fd153ba9e006a0a678d48c88195a2e18bd...,"{'input_tokens': 215, 'input_tokens_details': ...",None,r2__288451ff7a28713399c3fe9c,ai_data/deflect_creativity/openai/model_gpt-5....,batch_6a0a66ff75148190916b217bd42d0379


In [45]:
expected_round2 = expected_round1
actual_round2 = len(round2_df)

print("Expected Round 2 rows:", expected_round2)
print("Actual Round 2 rows:  ", actual_round2)

display(round2_df.groupby(["task_id", "strategy", "condition", "status"]).size().reset_index(name="n"))

if actual_round2 != expected_round2:
    print("WARNING: Row count mismatch. Inspect errors before compiling.")

if (round2_df["status"] != "success").any():
    print("WARNING: Some Round 2 calls failed or returned empty text.")
    display(round2_df[round2_df["status"] != "success"].head(20))
else:
    print("Round 2 looks complete.")

Expected Round 2 rows: 5400
Actual Round 2 rows:   5400


,task_id,strategy,condition,status,n
0,aut_button,diverge,base,success,150
1,aut_button,diverge,dyad,success,150
2,aut_button,diverge,triad,success,150
3,aut_button,vanilla,base,success,150
4,aut_button,vanilla,dyad,success,150
5,aut_button,vanilla,triad,success,150
6,aut_shoe,diverge,base,success,150
7,aut_shoe,diverge,dyad,success,150
8,aut_shoe,diverge,triad,success,150
9,aut_shoe,vanilla,base,success,150


Round 2 looks complete.


In [46]:
def normalize_for_analysis(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Guarantee columns exist across rounds.
    for col in [
        "self_round1_request_key",
        "peer_round1_agent_ids",
        "self_round1_text",
        "peer_round1_texts_json",
    ]:
        if col not in out.columns:
            out[col] = None

    keep_cols = [
        "provider",
        "model",
        "round",
        "task_id",
        "task_family",
        "strategy",
        "condition",
        "group_id",
        "agent_id",
        "agent_index",
        "request_key",
        "status",
        "text",
        "temperature",
        "max_output_tokens",
        "self_round1_request_key",
        "peer_round1_agent_ids",
        "self_round1_text",
        "peer_round1_texts_json",
        "provider_response_id",
        "usage",
        "error",
        "batch_id",
        "batch_custom_id",
        "batch_output_file",
        "parsed_at_utc",
    ]

    existing_keep_cols = [c for c in keep_cols if c in out.columns]
    out = out[existing_keep_cols].copy()

    out["text_clean"] = out["text"].map(clean_model_text)
    out["is_success"] = out["status"].eq("success")

    return out


round1_analysis_df = normalize_for_analysis(round1_df)
round2_analysis_df = normalize_for_analysis(round2_df)

full_long_df = pd.concat([round1_analysis_df, round2_analysis_df], ignore_index=True)

sort_cols = ["task_id", "strategy", "condition", "group_id", "agent_index", "round"]
full_long_df = full_long_df.sort_values(sort_cols).reset_index(drop=True)

print(full_long_df.shape)
display(full_long_df.groupby(["round", "task_id", "strategy", "condition", "status"]).size().reset_index(name="n").head(30))

timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
full_csv_path = DIRS["compiled"] / f"{PROVIDER}__{MODEL_NAME}__deflect_creativity__full_long__{timestamp}.csv"
full_pkl_path = DIRS["compiled"] / f"{PROVIDER}__{MODEL_NAME}__deflect_creativity__full_long__{timestamp}.pkl"

full_long_df.to_csv(full_csv_path, index=False)
full_long_df.to_pickle(full_pkl_path)

print(full_csv_path)
print(full_pkl_path)

(10800, 28)


,round,task_id,strategy,condition,status,n
0,1,aut_button,diverge,base,success,150
1,1,aut_button,diverge,dyad,success,150
2,1,aut_button,diverge,triad,success,150
3,1,aut_button,vanilla,base,success,150
4,1,aut_button,vanilla,dyad,success,150
5,1,aut_button,vanilla,triad,success,150
6,1,aut_shoe,diverge,base,success,150
7,1,aut_shoe,diverge,dyad,success,150
8,1,aut_shoe,diverge,triad,success,150
9,1,aut_shoe,vanilla,base,success,150


ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/03_compiled/openai__gpt-5.4__deflect_creativity__full_long__20260517_215356.csv
ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/03_compiled/openai__gpt-5.4__deflect_creativity__full_long__20260517_215356.pkl


In [47]:
r1_small = full_long_df[full_long_df["round"].eq(1)].copy()
r2_small = full_long_df[full_long_df["round"].eq(2)].copy()

merge_keys = ["provider", "model", "task_id", "task_family", "strategy", "condition", "group_id", "agent_id", "agent_index"]

wide_df = r1_small[merge_keys + ["request_key", "status", "text_clean", "batch_id", "usage"]].rename(
    columns={
        "request_key": "round1_request_key",
        "status": "round1_status",
        "text_clean": "round1_text",
        "batch_id": "round1_batch_id",
        "usage": "round1_usage",
    }
).merge(
    r2_small[merge_keys + [
        "request_key",
        "status",
        "text_clean",
        "batch_id",
        "usage",
        "self_round1_request_key",
        "peer_round1_agent_ids",
        "self_round1_text",
        "peer_round1_texts_json",
    ]].rename(
        columns={
            "request_key": "round2_request_key",
            "status": "round2_status",
            "text_clean": "round2_text",
            "batch_id": "round2_batch_id",
            "usage": "round2_usage",
        }
    ),
    on=merge_keys,
    how="outer",
    validate="one_to_one",
)

wide_df = wide_df.sort_values(["task_id", "strategy", "condition", "group_id", "agent_index"]).reset_index(drop=True)

wide_csv_path = DIRS["compiled"] / f"{PROVIDER}__{MODEL_NAME}__deflect_creativity__ego_wide__{timestamp}.csv"
wide_pkl_path = DIRS["compiled"] / f"{PROVIDER}__{MODEL_NAME}__deflect_creativity__ego_wide__{timestamp}.pkl"

wide_df.to_csv(wide_csv_path, index=False)
wide_df.to_pickle(wide_pkl_path)

print(wide_df.shape)
print(wide_csv_path)
print(wide_pkl_path)
wide_df.head()

(5400, 23)
ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/03_compiled/openai__gpt-5.4__deflect_creativity__ego_wide__20260517_215356.csv
ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/03_compiled/openai__gpt-5.4__deflect_creativity__ego_wide__20260517_215356.pkl


,provider,model,task_id,task_family,strategy,condition,group_id,agent_id,agent_index,round1_request_key,...,round1_usage,round2_request_key,round2_status,round2_text,round2_batch_id,round2_usage,self_round1_request_key,peer_round1_agent_ids,self_round1_text,peer_round1_texts_json
0,openai,gpt-5.4,aut_button,aut,diverge,base,base_001,base_001__a1,1,r1__eeeff5348ceea507a0291cd3,...,"{'input_tokens': 163, 'input_tokens_details': ...",r2__aa96f4ec51cf9b0a2948445a,success,Sew a button inside a lampshade as a discreet ...,batch_6a0a66ff75148190916b217bd42d0379,"{'input_tokens': 221, 'input_tokens_details': ...",r1__eeeff5348ceea507a0291cd3,NaN,A button can be glued under a wobbly table leg...,[]
1,openai,gpt-5.4,aut_button,aut,diverge,base,base_002,base_002__a1,1,r1__2ecc55303e06d1b1b983de33,...,"{'input_tokens': 163, 'input_tokens_details': ...",r2__be4c9723220ca51d483b00f8,success,A button can be embedded in candle wax as a hi...,batch_6a0a66ff75148190916b217bd42d0379,"{'input_tokens': 216, 'input_tokens_details': ...",r1__2ecc55303e06d1b1b983de33,NaN,A button can serve as a tiny loom for weaving ...,[]
2,openai,gpt-5.4,aut_button,aut,diverge,base,base_003,base_003__a1,1,r1__bdca0bc782d460428d40b829,...,"{'input_tokens': 163, 'input_tokens_details': ...",r2__b607b909781a1c9109e5ee1a,success,A button tied to a houseplant stem works as a ...,batch_6a0a66ff75148190916b217bd42d0379,"{'input_tokens': 225, 'input_tokens_details': ...",r1__bdca0bc782d460428d40b829,NaN,A button sewn inside a shirt cuff can serve as...,[]
3,openai,gpt-5.4,aut_button,aut,diverge,base,base_004,base_004__a1,1,r1__61433013f153cb6bbda73844,...,"{'input_tokens': 163, 'input_tokens_details': ...",r2__0309eb4d8b21f5934d32f4ac,success,A button can be glued under a wobbly picture f...,batch_6a0a66ff75148190916b217bd42d0379,"{'input_tokens': 219, 'input_tokens_details': ...",r1__61433013f153cb6bbda73844,NaN,A button can serve as a tiny paint palette for...,[]
4,openai,gpt-5.4,aut_button,aut,diverge,base,base_005,base_005__a1,1,r1__8b386a285bd41aea4087e87b,...,"{'input_tokens': 163, 'input_tokens_details': ...",r2__288451ff7a28713399c3fe9c,success,Glue a button inside a flowerpot as a hidden r...,batch_6a0a66ff75148190916b217bd42d0379,"{'input_tokens': 215, 'input_tokens_details': ...",r1__8b386a285bd41aea4087e87b,NaN,Use a button as a tiny paint palette for mixin...,[]


In [48]:
def word_count(text: str) -> int:
    if not isinstance(text, str):
        return 0
    return len(re.findall(r"\b[\w'-]+\b", text))


def sentence_count_rough(text: str) -> int:
    if not isinstance(text, str):
        return 0
    # Rough validation only; final analysis can use a better sentence splitter.
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    parts = [p for p in parts if p.strip()]
    return len(parts)


validation_df = full_long_df.copy()
validation_df["word_count"] = validation_df["text_clean"].map(word_count)
validation_df["rough_sentence_count"] = validation_df["text_clean"].map(sentence_count_rough)

# Slogan word-count check.
slogan_violations = validation_df[
    validation_df["task_family"].eq("slogan")
    & validation_df["is_success"]
    & validation_df["word_count"].gt(6)
].copy()

# Story sentence-count rough check.
story_sentence_violations = validation_df[
    validation_df["task_family"].eq("story")
    & validation_df["is_success"]
    & validation_df["rough_sentence_count"].ne(8)
].copy()

print("Slogan >6-word violations:", len(slogan_violations))
display(slogan_violations[["round", "task_id", "strategy", "condition", "agent_id", "text_clean", "word_count"]].head(20))

print("Story rough sentence-count violations:", len(story_sentence_violations))
display(story_sentence_violations[["round", "task_id", "strategy", "condition", "agent_id", "text_clean", "rough_sentence_count"]].head(20))

validation_csv_path = DIRS["compiled"] / f"{PROVIDER}__{MODEL_NAME}__deflect_creativity__validation_flags__{timestamp}.csv"
validation_df.to_csv(validation_csv_path, index=False)
validation_csv_path

Slogan >6-word violations: 0


,round,task_id,strategy,condition,agent_id,text_clean,word_count


Story rough sentence-count violations: 361


,round,task_id,strategy,condition,agent_id,text_clean,rough_sentence_count
7208,1,story_jungle,diverge,base,base_005__a1,"By the time the map's ink began to glow, Mara ...",7
7212,1,story_jungle,diverge,base,base_007__a1,"By the time the map's ink began to glow, Lina ...",7
7229,2,story_jungle,diverge,base,base_015__a1,"On the day the river vanished, seventeen-year-...",7
7231,2,story_jungle,diverge,base,base_016__a1,The jungle first announced itself with a rain ...,7
7243,2,story_jungle,diverge,base,base_022__a1,"On the third night after the storm, Milo heard...",7
7249,2,story_jungle,diverge,base,base_025__a1,"On the third day of the monsoon, Suri followed...",7
7289,2,story_jungle,diverge,base,base_045__a1,"On the third night of our survey trip, the jun...",7
7310,1,story_jungle,diverge,base,base_056__a1,"At dawn, Mira followed a map inked on orchid p...",7
7316,1,story_jungle,diverge,base,base_059__a1,"At dawn, Mira followed a broken line of blue b...",7
7334,1,story_jungle,diverge,base,base_068__a1,"At dawn, Mara followed a map inked on a strip ...",7


PosixPath('ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/03_compiled/openai__gpt-5.4__deflect_creativity__validation_flags__20260517_215356.csv')

In [49]:
final_manifest = {
    "run_id": RUN_ID,
    "data_root": str(DATA_ROOT),
    "provider": PROVIDER,
    "model": MODEL_NAME,
    "round1_batch_info": round1_batch_info,
    "round2_batch_info": round2_batch_info,
    "round1_parse_summary": round1_parse_summary,
    "round2_parse_summary": round2_parse_summary,
    "compiled_long_csv": str(full_csv_path),
    "compiled_long_pkl": str(full_pkl_path),
    "compiled_wide_csv": str(wide_csv_path),
    "compiled_wide_pkl": str(wide_pkl_path),
    "validation_csv": str(validation_csv_path),
    "completed_at_utc": now_iso(),
}

final_manifest_path = DIRS["compiled"] / f"{PROVIDER}__{MODEL_NAME}__deflect_creativity__final_manifest__{timestamp}.json"
write_json(final_manifest_path, final_manifest)

final_manifest_path

PosixPath('ai_data/deflect_creativity/openai/model_gpt-5.4/run_20260517_204221__2229089f/03_compiled/openai__gpt-5.4__deflect_creativity__final_manifest__20260517_215356.json')